# Retrieval for RAG with LangChain

Epzero as a LangChain vector store, with a local sentence-embedding model. Retrieval only: the retrieved
passages are assembled into a prompt for whichever LLM you use.

```bash
pip install "epzero[langchain]" sentence-transformers
```

In [1]:
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer

from epzero.langchain import EpzeroVectorStore

class LocalEmbeddings(Embeddings):
    def __init__(self, name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(name)
    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()
    def embed_query(self, text):
        return self.model.encode(text, normalize_embeddings=True).tolist()

embeddings = LocalEmbeddings()

In [2]:
DOCS = [
    ("To reset your password, open Settings, choose Security and select Reset password.", "account"),
    ("Two-factor authentication can be turned on under Settings, Security, Two-step sign-in.", "account"),
    ("If you forgot your username, use the Forgot username link on the sign-in page.", "account"),
    ("Invoices are emailed on the first of each month and can be downloaded under Billing.", "billing"),
    ("To change your payment card, go to Billing and select Payment methods.", "billing"),
    ("Refunds for annual plans are prorated if requested within 30 days.", "billing"),
    ("You can cancel your subscription at any time from the Billing page.", "billing"),
    ("The mobile app syncs automatically when you are connected to Wi-Fi.", "app"),
    ("Offline mode keeps your last 100 documents available without a connection.", "app"),
    ("Export your data as CSV or JSON from Settings, Data, Export.", "data"),
    ("Deleted files stay in the trash for 30 days before they are removed for good.", "data"),
    ("Shared folders can be given view-only or edit access per person.", "sharing"),
    ("Links to shared files can be set to expire after a chosen date.", "sharing"),
    ("Our API is rate limited to 100 requests per minute per key.", "developers"),
    ("Webhooks notify your server when a document is created or updated.", "developers"),
    ("API keys are created and revoked under Settings, Developers.", "developers"),
]

In [3]:
store = EpzeroVectorStore.from_texts([t for t, _ in DOCS], embeddings,
                                   metadatas=[{"topic": topic} for _, topic in DOCS], metric="cosine")
for doc, distance in store.similarity_search_with_score("I can't log in, how do I change my password?", k=3):
    print(f"{distance:.3f}  [{doc.metadata['topic']}]  {doc.page_content}")

0.351  [account]  To reset your password, open Settings, choose Security and select Reset password.
0.583  [account]  If you forgot your username, use the Forgot username link on the sign-in page.
0.657  [account]  Two-factor authentication can be turned on under Settings, Security, Two-step sign-in.


Metadata filters run in the engine; maximal marginal relevance trades similarity for variety.

In [4]:
for doc in store.similarity_search("how do I get my money back", k=2, filter={"topic": "billing"}):
    print("billing:", doc.page_content)
for doc in store.max_marginal_relevance_search("sharing files with my team", k=3, fetch_k=8):
    print("mmr:", doc.page_content)

billing: Refunds for annual plans are prorated if requested within 30 days.
billing: To change your payment card, go to Billing and select Payment methods.
mmr: Shared folders can be given view-only or edit access per person.
mmr: Export your data as CSV or JSON from Settings, Data, Export.
mmr: Deleted files stay in the trash for 30 days before they are removed for good.


## Into a prompt

In [5]:
question = "Can I get a refund if I cancel my yearly plan?"
retriever = store.as_retriever(search_kwargs={"k": 3})
context = "\n".join(f"- {d.page_content}" for d in retriever.invoke(question))
prompt = f"Answer from the context only.\n\nContext:\n{context}\n\nQuestion: {question}"
print(prompt)

Answer from the context only.

Context:
- Refunds for annual plans are prorated if requested within 30 days.
- You can cancel your subscription at any time from the Billing page.
- To change your payment card, go to Billing and select Payment methods.

Question: Can I get a refund if I cancel my yearly plan?
